# Session 07 practical lab: Classification challenge for top-price opportunity scoring

> Warning: This notebook is a teaching lab. The production baseline notebook remains `05_classification_audi_a3_germany.ipynb`.

This lab runs directly from processed CSV files. It does not require cloud credentials, BigQuery access, or warehouse writes.

Head of Data 101 uses one central idea: **the pipeline is the product**.

In this lab, classification predicts whether a listing belongs to the external marketplace `top-price` category. The model output is `probability_top_price`. The business artifact is a review queue, not an automatic acquisition decision.

Course narrative boundary:

- `price_label` is an observed marketplace label.
- `top_price` is the binary target derived from `price_label`.
- `top_price = 1` when `price_label == "top-price"`; otherwise `top_price = 0`.
- Neither `price_label` nor `top_price` can be used as model input features.
- Using the target, or a direct source of the target, as an input feature is target leakage.
- Regression remains separate and predicts `expected_price_eur`.
- BI later combines actual price, expected-price gap, and top-price probability.

For beginner clarity, the first classifier does not use `actual_price_eur` or `price_eur` as a feature. Price can be discussed later as an advanced feature because it may be too close to the marketplace price-tier logic.


## How to work

This is not a pure coding course and it is not a theoretical machine learning course. Use AI-assisted coding when useful, but make sure you understand and can defend every result.

For each challenge, read the business reason, adapt or write the code, inspect the output, and write a short business interpretation in English.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.3f}".format)
plt.style.use("default")


## Challenge 0 - CSV setup and classification data contract

**Suggested time:** 10 minutes

**Business reason:**  
Before modeling, a data scientist must know exactly which dataset contract is being consumed and where the target comes from.

**Objective:**  
Load the processed CSV, validate required columns, create `listing_id` if needed, create `top_price` from `price_label` if needed, and prepare a modeling dataframe.

**Data source priority:**
- First: preferred full Session 07 sample CSV in `data/processed/`, if available locally.
- Second: the same filename in `data/sample/processed/`.
- Third: most recent local processed CSV in `data/processed/*.csv`.
- Last resort: any `data/**/*.csv` that already contains the required classification columns.

The lab depends on the processed CSV schema, not on a particular timestamped file.

**Required minimum columns:** `price_label`, `mileage_km`, `age_years`, `power_hp`.

**Preferred optional columns:** `listing_id`, `make`, `model`, `brand`, `fuel_type`, `listing_country`, `price_eur`, `actual_price_eur`, `registration_year`, `registration_month`, `price_outlier_iqr`, `mileage_outlier_iqr`, `power_outlier_iqr`, `logical_issue`.

**Expected output format:**
- Selected CSV path.
- Row count before and after validity filtering.
- Available columns.
- Target definition summary.
- First 5 rows.
- Missing values summary.

**Reflection questions:**
- Which CSV did the notebook load?
- Is `top_price` already present or created from `price_label`?
- Why should target definition fail loudly rather than be guessed?


In [ ]:
REQUIRED_COLUMNS = ["price_label", "mileage_km", "age_years", "power_hp"]
SESSION_07_FULL_CSV = "autoscout24_listings_processed_audi_a3_germany_20251228_205210.csv"
OPTIONAL_COLUMNS = [
    "listing_id",
    "make",
    "model",
    "brand",
    "fuel_type",
    "listing_country",
    "price_eur",
    "actual_price_eur",
    "registration_year",
    "registration_month",
    "price_outlier_iqr",
    "mileage_outlier_iqr",
    "power_outlier_iqr",
    "logical_issue",
]
LEAKAGE_COLUMNS = ["price_label", "price_label_id", "top_price"]


def find_repo_root(start):
    for path in [start] + list(start.parents):
        if (path / ".git").exists() or (path / "config" / "project_config.yaml").exists():
            return path
    return start


def load_project_config(config_path):
    config = {}
    if not config_path.exists():
        return config
    for raw_line in config_path.read_text(encoding="utf-8-sig").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or ":" not in line:
            continue
        key, value = line.split(":", 1)
        key = key.strip()
        value = value.strip()
        if value.startswith(("'", '"')) and value.endswith(("'", '"')):
            value = value[1:-1]
        elif value.lower() in ("true", "false"):
            value = value.lower() == "true"
        else:
            try:
                value = int(value)
            except ValueError:
                try:
                    value = float(value)
                except ValueError:
                    pass
        config[key] = value
    return config


def relative_path(path):
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def csv_missing_required_columns(path):
    try:
        columns = pd.read_csv(path, nrows=0).columns
    except Exception as exc:
        return REQUIRED_COLUMNS, str(exc)
    missing = [column for column in REQUIRED_COLUMNS if column not in columns]
    return missing, None


def csv_contains_required_columns(path):
    missing, error = csv_missing_required_columns(path)
    return not missing and error is None


def newest_csv_in_folder(folder):
    if not folder.exists():
        return None
    csv_files = [path for path in folder.glob("*.csv") if path.is_file()]
    if not csv_files:
        return None
    return max(csv_files, key=lambda path: path.stat().st_mtime)


def newest_data_csv_with_required_columns(data_root):
    if not data_root.exists():
        return None
    candidates = []
    for path in data_root.rglob("*.csv"):
        if path.is_file() and csv_contains_required_columns(path):
            candidates.append(path)
    if not candidates:
        return None
    return max(candidates, key=lambda path: path.stat().st_mtime)


PROJECT_ROOT = find_repo_root(Path.cwd())
PROJECT_CONFIG = load_project_config(PROJECT_ROOT / "config" / "project_config.yaml")
RANDOM_SEED = int(PROJECT_CONFIG.get("random_state", 42))
np.random.seed(RANDOM_SEED)

make_scope = str(PROJECT_CONFIG.get("make", "Audi")).strip()
model_scope = str(PROJECT_CONFIG.get("model", "A3")).strip()
country_scope = str(PROJECT_CONFIG.get("country", "Germany")).strip()
TAG = f"{make_scope}_{model_scope}_{country_scope}".lower().replace(" ", "_")

processed_folder = PROJECT_ROOT / str(PROJECT_CONFIG.get("processed_data_path", "data/processed"))
sample_processed_folder = PROJECT_ROOT / "data" / "sample" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

preferred_csv_paths = [
    ("Preferred full Session 07 sample CSV (local)", processed_folder / SESSION_07_FULL_CSV),
    ("Repo-visible fallback sample", sample_processed_folder / SESSION_07_FULL_CSV),
]

selected_csv_path = None
selected_source = None
rejected_candidates = []

for source_label, candidate in preferred_csv_paths:
    if candidate.exists():
        missing, error = csv_missing_required_columns(candidate)
        if not missing and error is None:
            selected_csv_path = candidate
            selected_source = source_label
            break
        rejected_candidates.append((source_label, candidate, missing, error))

if selected_csv_path is None:
    candidate = newest_csv_in_folder(processed_folder)
    if candidate is not None:
        missing, error = csv_missing_required_columns(candidate)
        if not missing and error is None:
            selected_csv_path = candidate
            selected_source = "Latest local processed CSV"
        else:
            rejected_candidates.append(("Latest local processed CSV", candidate, missing, error))

if selected_csv_path is None:
    selected_csv_path = newest_data_csv_with_required_columns(PROJECT_ROOT / "data")
    selected_source = "Last-resort scan for any CSV with required classification columns"

if selected_csv_path is None:
    details = []
    for source_label, path, missing, error in rejected_candidates:
        if error:
            details.append(f"{source_label}: {relative_path(path)} could not be read ({error})")
        else:
            details.append(f"{source_label}: {relative_path(path)} missing {missing}")
    detail_text = "\n".join(details)
    raise FileNotFoundError(
        "No processed CSV with the required classification columns was found. "
        "Run Notebook 02 to create data/processed/*.csv, or use data/sample/processed/.\n"
        + detail_text
    )

df_raw = pd.read_csv(selected_csv_path)
row_count_before = len(df_raw)
missing_required = [column for column in REQUIRED_COLUMNS if column not in df_raw.columns]
if missing_required:
    raise ValueError(
        "Missing required classification columns: "
        + ", ".join(missing_required)
        + ". Required columns are: "
        + ", ".join(REQUIRED_COLUMNS)
    )

missing_optional = [column for column in OPTIONAL_COLUMNS if column not in df_raw.columns]
df = df_raw.copy()
created_listing_id = "listing_id" not in df.columns
had_top_price = "top_price" in df.columns

for column in ["mileage_km", "age_years", "power_hp", "price_eur", "actual_price_eur"]:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

df["price_label"] = df["price_label"].astype("string").str.strip().str.lower()
top_price_from_label = df["price_label"].eq("top-price").astype(int)

if had_top_price:
    top_price_text = df["top_price"].astype("string").str.strip().str.lower()
    top_price_text_map = top_price_text.map({"true": 1, "false": 0, "yes": 1, "no": 0, "1": 1, "0": 0})
    existing_top_price = pd.to_numeric(df["top_price"], errors="coerce")
    existing_top_price = existing_top_price.where(existing_top_price.notna(), top_price_text_map)
    invalid_existing_mask = existing_top_price.notna() & ~existing_top_price.isin([0, 1])
    if invalid_existing_mask.any():
        raise ValueError(
            "Existing top_price contains values other than 0/1 for "
            f"{int(invalid_existing_mask.sum())} rows. Fix the target definition before modeling."
        )
    mismatch_mask = existing_top_price.notna() & existing_top_price.astype("Int64").ne(top_price_from_label)
    if mismatch_mask.any():
        raise ValueError(
            "Existing top_price does not match price_label == 'top-price' for "
            f"{int(mismatch_mask.sum())} rows. Fix the target definition before modeling."
        )
    top_price_status = "present and validated against price_label"
else:
    top_price_status = "created from price_label"

df["top_price"] = top_price_from_label

valid_modeling_rows = (
    df["price_label"].notna()
    & df["mileage_km"].gt(0)
    & df["age_years"].ge(0)
    & df["power_hp"].gt(0)
)
rows_removed = int((~valid_modeling_rows).sum())
df = df.loc[valid_modeling_rows].reset_index(drop=True)

if created_listing_id:
    df.insert(0, "listing_id", np.arange(1, len(df) + 1))

if df.empty:
    raise ValueError(
        "The selected CSV has no valid modeling rows after filtering. Required values must satisfy "
        "price_label present, mileage_km > 0, age_years >= 0, and power_hp > 0."
    )

if df["top_price"].nunique() < 2:
    raise ValueError(
        "The target top_price has only one class after filtering. Classification requires both 0 and 1 examples."
    )

numeric_features = ["mileage_km", "age_years", "power_hp"]
categorical_features = []
if "fuel_type" in df.columns and df["fuel_type"].nunique(dropna=True) > 1:
    categorical_features.append("fuel_type")

feature_columns = numeric_features + categorical_features
excluded_feature_columns = [column for column in LEAKAGE_COLUMNS + ["actual_price_eur", "price_eur"] if column in df.columns]

print("Selected CSV:", relative_path(selected_csv_path))
print("Source priority:", selected_source)
print("Configured scope:", make_scope, model_scope, country_scope)
print("Rows before filtering:", row_count_before)
print("Rows after filtering:", len(df))
print("Rows removed by required-value filter:", rows_removed)
print("Available columns:")
print(", ".join(df.columns))
print("Target definition: top_price = 1 when price_label == 'top-price', otherwise 0")
print("top_price status:", top_price_status)
print("Main feature columns:", feature_columns)
print("Excluded from model features because of leakage or beginner clarity:", excluded_feature_columns)
if missing_optional:
    print("Optional columns not found in this CSV:", ", ".join(missing_optional))

print("\nFirst 5 rows:")
display(df.head())

print("Missing values summary:")
missing_summary = df.isna().sum().sort_values(ascending=False).to_frame("missing_values")
display(missing_summary[missing_summary["missing_values"] > 0])


## Challenge 1 - Understand the label before modeling

**Suggested time:** 20 minutes

**Business reason:**  
Classification starts with the label. A model cannot be interpreted if the positive class is not understood.

**Objective:**  
Inspect `price_label` and `top_price` distribution.

**Expected tasks:**
- Count values of `price_label`.
- Count and percentage of `top_price`.
- Plot class distribution.
- Identify whether the problem is balanced or imbalanced.

**Expected output format:**
- Frequency table.
- Percentage table.
- Bar chart.
- Short interpretation.

**Reflection questions:**
- How common is the positive class?
- Would accuracy be misleading here?
- What would a naive model achieve by always predicting the majority class?


In [ ]:
# Challenge 1 scaffold.
# Start with the target before you train any model.

# Your code here
# price_label_counts = df["price_label"].value_counts(dropna=False)
# price_label_percent = df["price_label"].value_counts(normalize=True, dropna=False) * 100
# display(pd.DataFrame({"count": price_label_counts, "percent": price_label_percent}))
#
# target_counts = df["top_price"].value_counts().sort_index()
# target_percent = df["top_price"].value_counts(normalize=True).sort_index() * 100
# display(pd.DataFrame({"count": target_counts, "percent": target_percent}))
#
# plt.figure(figsize=(6, 4))
# plt.bar(["not top-price", "top-price"], [target_counts.get(0, 0), target_counts.get(1, 0)])
# plt.title("Top-price class distribution")
# plt.ylabel("Listings")
# plt.tight_layout()
# plt.show()
#
# Write one short interpretation below the output.


### Your label interpretation

1. Positive class share:
2. Is the target balanced or imbalanced?
3. Would accuracy alone be enough? Why?
4. Majority-class baseline expectation:


## Challenge 2 - Naive baseline and first evaluation logic

**Suggested time:** 20 minutes

**Business reason:**  
Before training a classifier, we need a baseline. A complex model must beat a simple rule.

**Objective:**  
Build a naive baseline that always predicts the majority class.

**Expected tasks:**
- Split `X` and `y` using stratified train-test split.
- Evaluate the majority-class baseline on the test set.
- Report accuracy, precision, recall, F1, and confusion matrix.

**Expected output format:**
- Baseline metrics table.
- Confusion matrix.
- One paragraph explaining why accuracy alone is dangerous.

**Reflection questions:**
- Did the baseline detect any top-price listings?
- Which metric shows this failure most clearly?
- What business risk appears if the review queue misses all positives?


In [ ]:
# Challenge 2 scaffold.
# Keep price_label, price_label_id, and top_price out of X. They are leakage.

# Your code here
# X = df[feature_columns].copy()
# y = df["top_price"].astype(int)
#
# X_train, X_test, y_train, y_test = train_test_split(
#     X,
#     y,
#     test_size=0.20,
#     random_state=RANDOM_SEED,
#     stratify=y,
# )
#
# majority_class = y_train.mode()[0]
# y_pred_baseline = np.repeat(majority_class, len(y_test))
#
# Calculate accuracy, precision, recall, F1, and confusion matrix.
# Use zero_division=0 for precision and recall.


### Your baseline conclusion

1. Did the baseline detect any `top_price` listings?
2. Which metric makes that visible?
3. What business risk does this create?


## Challenge 3 - Logistic regression as an interpretable classifier

**Suggested time:** 30 minutes

**Business reason:**  
A professional data scientist starts with a transparent model before moving to flexible models.

**Objective:**  
Train Logistic Regression using a clean feature set.

**Main feature set:**
- `mileage_km`
- `age_years`
- `power_hp`
- `fuel_type` if available

**Do not include:**
- `price_label`
- `price_label_id`
- `top_price`
- `actual_price_eur` or `price_eur` in the main first model

Those exclusions matter because `price_label` and `top_price` would leak the answer into the model. Price is kept out of the beginner model because it may be too close to the marketplace price-tier logic.

**Expected tasks:**
- Define numeric and categorical columns.
- Build a preprocessing pipeline.
- Fit Logistic Regression.
- Evaluate accuracy, precision, recall, F1, ROC-AUC.
- Show confusion matrix.
- Show top coefficients if feasible.

**Expected output format:**
- Metrics table.
- Confusion matrix.
- Coefficient interpretation where possible.
- Business interpretation.

**Reflection questions:**
- Which variables appear most associated with top-price classification?
- Does the model detect positives?
- Is the model useful as a first review filter?


In [ ]:
# Challenge 3 scaffold.
# Logistic Regression needs numeric scaling and categorical encoding.

# Your code here
# def make_one_hot_encoder():
#     try:
#         return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
#     except TypeError:
#         return OneHotEncoder(handle_unknown="ignore", sparse=False)
#
# numeric_transformer = Pipeline([
#     ("imputer", SimpleImputer(strategy="median")),
#     ("scaler", StandardScaler()),
# ])
# categorical_transformer = Pipeline([
#     ("imputer", SimpleImputer(strategy="most_frequent")),
#     ("encoder", make_one_hot_encoder()),
# ])
#
# transformers = [("numeric", numeric_transformer, numeric_features)]
# if categorical_features:
#     transformers.append(("categorical", categorical_transformer, categorical_features))
#
# preprocessor = ColumnTransformer(transformers=transformers)
# minority_share = y_train.value_counts(normalize=True).min()
# class_weight = "balanced" if minority_share < 0.30 else None
#
# logreg_model = Pipeline([
#     ("preprocess", preprocessor),
#     ("model", LogisticRegression(max_iter=2000, class_weight=class_weight)),
# ])
#
# Fit, predict, evaluate, and inspect coefficients.


### Your Logistic Regression conclusion

1. Which variables seem most associated with `top_price`?
2. Did the model detect positives?
3. Would you use it as a first review filter?


## Challenge 4 - Tree-based models: controlled increase in flexibility

**Suggested time:** 35 minutes

**Business reason:**  
Flexible models can capture non-linear patterns, but they may be harder to explain.

**Objective:**  
Compare Logistic Regression with Decision Tree and Random Forest.

**Models:**
- `LogisticRegression(max_iter=2000, class_weight="balanced" when needed)`
- `DecisionTreeClassifier(max_depth=4, random_state=RANDOM_SEED, class_weight="balanced" when needed)`
- `RandomForestClassifier(n_estimators=100, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1, class_weight="balanced_subsample" when needed)`

HistGradientBoostingClassifier is an instructor extension, not a core student burden for this lab.

**Expected tasks:**
- Train all models on the same train/test split.
- Compare metrics.
- Plot confusion matrices.
- Plot feature importance for Random Forest if available.

**Expected output format:**
- Model comparison table.
- Confusion matrices.
- Feature importance chart.
- Short business interpretation.

**Reflection questions:**
- Which model improves recall?
- Which model protects precision?
- Is the gain large enough to justify complexity?
- Which model would be easiest to explain to a committee?


In [ ]:
# Challenge 4 scaffold.
# Train each model on the same X_train, X_test, y_train, y_test split.

# Your code here
# tree_class_weight = "balanced" if use_class_weight else None
# forest_class_weight = "balanced_subsample" if use_class_weight else None
#
# candidate_models = {
#     "Logistic Regression": LogisticRegression(max_iter=2000, class_weight=logreg_class_weight),
#     "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_SEED, class_weight=tree_class_weight),
#     "Random Forest": RandomForestClassifier(
#         n_estimators=100,
#         max_depth=8,
#         random_state=RANDOM_SEED,
#         n_jobs=-1,
#         class_weight=forest_class_weight,
#     ),
# }
#
# Loop through candidate_models.
# For each model, build a Pipeline, fit it, predict labels and probabilities, and store metrics.
# Plot confusion matrices and Random Forest feature importance if available.


### Your tree-model conclusion

1. Which model improves recall?
2. Which model protects precision?
3. Is the improvement large enough to justify complexity?
4. Which model would be easiest to explain to a committee?


## Challenge 5 - Threshold analysis: turning probability into policy

**Suggested time:** 30 minutes

**Business reason:**  
A classifier outputs probabilities. The threshold is an operational decision, not a model truth.

**Objective:**  
Analyze how different probability thresholds change precision, recall, and number of listings selected.

**Expected tasks:**
- Select the best model from Challenge 4.
- Generate predicted probabilities.
- Build a threshold table for thresholds such as 0.30, 0.40, 0.50, 0.60, 0.70.
- For each threshold, compute selected count, precision, recall, F1.

**Expected output format:**
- Threshold comparison table.
- Plot threshold vs precision and recall.
- Recommended threshold or Top-N policy.

**Reflection questions:**
- What threshold creates a manageable review queue?
- What is more costly here: false positives or false negatives?
- Why is 0.50 not automatically the correct threshold?


In [ ]:
# Challenge 5 scaffold.
# Pick a model from Challenge 4, then test several policy thresholds.

# Your code here
# selected_model_name = "..."
# selected_model = fitted_models[selected_model_name]
# y_score_selected = selected_model.predict_proba(X_test)[:, 1]
# thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]
# threshold_rows = []
#
# for threshold in thresholds:
#     y_pred_threshold = (y_score_selected >= threshold).astype(int)
#     # Calculate selected_count, precision, recall, and F1.
#     pass
#
# threshold_analysis_df = pd.DataFrame(threshold_rows)
# display(threshold_analysis_df)
# Plot threshold vs precision and recall.


### Your threshold conclusion

1. Which threshold creates a manageable review queue?
2. Which error is more costly here: false positives or false negatives?
3. Why is 0.50 not automatically correct?


## Challenge 6 - From classifier to BI-ready review queue

**Suggested time:** 25 minutes

**Business reason:**  
A classification model becomes useful only when it creates an actionable review artifact.

**Objective:**  
Train the selected model on all available data, generate `probability_top_price`, and create a review queue.

**Expected tasks:**
- Fit selected model on all valid rows.
- Generate probability scores.
- Create `classification_review_queue`.
- Include useful columns:
  - `listing_id`
  - `probability_top_price`
  - `predicted_top_price`
  - `price_label`
  - `top_price`
  - `mileage_km`
  - `age_years`
  - `power_hp`
  - `fuel_type` if available
  - `listing_country` if available
  - `price_eur` or `actual_price_eur` if available, but only as context in the output, not as a core feature unless explicitly discussed
- Sort by probability descending.
- Save CSV to `reports/{TAG}_classification_review_queue.csv`.
- Save model comparison CSV to `reports/{TAG}_classification_model_comparison.csv`.
- Save threshold analysis CSV to `reports/{TAG}_classification_threshold_analysis.csv`.

**Expected output format:**
- Final selected model name.
- Top 20 listings by `probability_top_price`.
- Recommended threshold or Top-N policy.
- One model risk.
- One recommended next improvement.
- One paragraph explaining how BI should consume the output.

**Reflection questions:**
- Which listings deserve analyst review first?
- Which predictions might be model artifacts?
- What information is still missing before making a real business decision?
- How should this output be shown in Power BI?


In [ ]:
# Challenge 6 scaffold.
# This is where the model becomes a BI-ready prioritization layer.

# Your code here
# final_model = selected_model
# final_model.fit(X, y)
# probability_top_price = final_model.predict_proba(X)[:, 1]
# predicted_top_price = (probability_top_price >= recommended_threshold).astype(int)
#
# output_columns = ["listing_id", "price_label", "top_price", "mileage_km", "age_years", "power_hp"]
# if "fuel_type" in df.columns:
#     output_columns.append("fuel_type")
# if "listing_country" in df.columns:
#     output_columns.append("listing_country")
# if "actual_price_eur" in df.columns:
#     output_columns.append("actual_price_eur")
# elif "price_eur" in df.columns:
#     output_columns.append("price_eur")
#
# classification_review_queue = df[output_columns].copy()
# classification_review_queue.insert(1, "probability_top_price", probability_top_price)
# classification_review_queue.insert(2, "predicted_top_price", predicted_top_price)
# classification_review_queue = classification_review_queue.sort_values("probability_top_price", ascending=False)
#
# Save the three requested CSV files under reports/.


### Your review queue conclusion

1. Final selected model:
2. Recommended threshold or Top-N policy:
3. Three listings that deserve analyst review first:
4. One model risk:
5. One recommended next improvement:
6. How should Power BI consume this output?


## Final discussion: what did we learn about classification as a prioritization layer?

Use these prompts for the final class discussion:

- Why did we inspect the target before training?
- Why did we build a naive baseline?
- Why is accuracy dangerous in imbalanced classification?
- Why is `price_label` allowed as the source of the target but not as a feature?
- What is leakage?
- How do precision and recall map to business costs?
- Why is threshold selection a policy decision?
- How does `probability_top_price` become a BI-ready prioritization layer?

Final message:

**A useful classifier does not end in a metric. It ends in a probability layer that helps people prioritize review, understand trade-offs, and make better decisions.**
